<a href="https://colab.research.google.com/github/amirgroup-codes/ProtoMech/blob/main/ProtoMech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p align="left">
  <img src="https://raw.githubusercontent.com/amirgroup-codes/ProtoMech/main/ProtoMech_Logo_Glow.svg"
       alt="ProtoMech"
       width="60%">
</p>

# ProtoMech: Protein Circuit Tracing via Cross-layer Transcoders</h1>

ProtoMech is a framework for discovering computational circuits in protein language models using cross-layer transcoders. This colab notebook is designed to produce the four files required for our [website](https://protmech.github.io/):
1. `activation_indices.json`
2. `seq.txt`
3. `top_activations.json`
4. `virtual_weights.json`

A link to the paper can be found [here](https://arxiv.org/abs/2602.XXXXX). We additionally provide our [code](https://github.com/amirgroup-codes/ProtoMech), [models](https://huggingface.co/anonymous-hf-user/ProtoMechModels), and [data](https://huggingface.co/datasets/anonymous-hf-user/ProtoMechData).

---

In [ ]:
# @title 0. Check GPU status and install dependencies
import torch
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU detected. Code may run extremely slowly.")
    print("----------------------------------------------------------------")
    print("TO FIX THIS:")
    print("1. Click 'Runtime' in the top menu.")
    print("2. Select 'Change runtime type'.")
    print("3. Under 'Hardware accelerator', select 'T4 GPU'.")
    print("4. Click 'Save' and then re-run this cell.")
    print("----------------------------------------------------------------")

import os
import sys
import shutil
import tarfile
import subprocess
from google.colab import files
from huggingface_hub import hf_hub_download
from IPython.display import clear_output
import pty
!pip install -q huggingface_hub torch pytorch-lightning fair-esm

GPU Detected: Tesla T4


In [ ]:
# @title 1. Clone ProtoMech and download models and data
REPO_URL = "https://github.com/amirgroup-codes/ProtoMech.git"
BRANCH = "main"
TARGET_DIR = "/content/ProtoMech"

# 1. Clean up or check for existing directory
if os.path.exists(TARGET_DIR):
    print(f"Directory {TARGET_DIR} already exists. Preparing for a fresh clone...")
    shutil.rmtree(TARGET_DIR)

# 2. Clone the repository (Public URL)
print(f"Cloning {REPO_URL}...")
result = os.system(f"git clone -q -b {BRANCH} {REPO_URL} {TARGET_DIR}")

if result == 0:
    print(f"Successfully cloned ProtoMech into {TARGET_DIR}")
else:
    print(f"ERROR: Failed to clone repository.")

# 3. Add to sys.path so the patches can find the files immediately
import sys
if TARGET_DIR not in sys.path:
    sys.path.append(TARGET_DIR)



# Download models
print('Downloading models...')
MODELS_DIR = os.path.join(TARGET_DIR, "models")
VISUALIZATION_DIR = os.path.join(TARGET_DIR, "visualization")
MODEL_REPO = "anonymous-hf-user/ProtoMechModels"
DATA_REPO = "anonymous-hf-user/ProtoMechData"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(VISUALIZATION_DIR, exist_ok=True)
os.makedirs(os.path.join(TARGET_DIR, "family_circuit/families"), exist_ok=True)
os.makedirs(os.path.join(TARGET_DIR, "function_circuit/functions"), exist_ok=True)
hf_hub_download(
    repo_id=MODEL_REPO,
    filename="esm2_t6_8M_UR50D.pt",
    repo_type="model",
    local_dir=MODELS_DIR,
    local_dir_use_symlinks=False
)
hf_hub_download(
    repo_id=MODEL_REPO,
    filename="CLT_L6_D3200/checkpoints/last.ckpt",
    repo_type="model",
    local_dir=MODELS_DIR,
    local_dir_use_symlinks=False
)

# Download datasets
hf_hub_download(
    repo_id=DATA_REPO,
    filename="top10_activations.pt",
    repo_type="dataset",
    local_dir=VISUALIZATION_DIR,
    local_dir_use_symlinks=False
)
def download_and_extract(filename, target_dir):
    print(f"Processing {filename}...")
    # 1. Download the tar.gz file
    tar_path = hf_hub_download(
        repo_id=DATA_REPO,
        filename=filename,
        repo_type="dataset",
        local_dir=TARGET_DIR # Download to root first
    )
    # 2. Extract into the specific target folder
    print(f"Extracting into {target_dir}...")
    try:
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=target_dir)
        print("Success.")
    except Exception as e:
        print(f"Error extracting {filename}: {e}")
download_and_extract("families.tar.gz", os.path.join(TARGET_DIR, "family_circuit/families"))
download_and_extract("functions.tar.gz", os.path.join(TARGET_DIR, "function_circuit/functions"))

if not os.path.exists(f"{MODELS_DIR}/esm2_t6_8M_UR50D.pt"): raise ValueError(f"ESM not found at {MODELS_DIR}/esm2_t6_8M_UR50D.pt, check download")
if not os.path.exists(f"{MODELS_DIR}/CLT_L6_D3200/checkpoints/last.ckpt"): raise ValueError(f"CLT not found at {MODELS_DIR}/CLT_L6_D3200/checkpoints/last.ckpt, check download")
if not os.path.exists(f"{VISUALIZATION_DIR}/top10_activations.pt"): raise ValueError(f"top10_activations.pt not found at {VISUALIZATION_DIR}/top10_activations.pt, check download")



# Patch code to get around PyTorch security updates and load checkpoints correctly
import os
REPO_ROOT = "/content/ProtoMech"
def patch_clt_circuit():
    """Specific fix for the multi-line call in clt_circuit.py"""
    path = os.path.join(REPO_ROOT, "circuit_utils/clt_circuit.py")
    if not os.path.exists(path): return
    with open(path, "r") as f: content = f.read()
    content = content.replace("load_from_checkpoint(weights_only=False, ", "load_from_checkpoint(")
    content = content.replace(", weights_only=False)", ")")
    target = "esm2_weight=self.esm_weights\n            )"
    fix = "esm2_weight=self.esm_weights, weights_only=False\n            )"
    if target in content and "weights_only=False" not in content:
        content = content.replace(target, fix)
        with open(path, "w") as f: f.write(content)
patch_clt_circuit()

def patch_file(file_path, find_str, replace_str, strategy_name="Standard Replace"):
    """Helper to find and replace content in a file."""
    if not os.path.exists(file_path):
        print(f"Warning: {os.path.basename(file_path)} not found at {file_path}")
        return False
    with open(file_path, "r") as f:
        content = f.read()
    if replace_str in content:
        return True
    if find_str in content:
        new_content = content.replace(find_str, replace_str)
        with open(file_path, "w") as f:
            f.write(new_content)
        return True
    else:
        return False

# Patch visualization scripts and edge_weights.py with weights_only=False
viz_files = [
    os.path.join(REPO_ROOT, "visualization/circuit_analysis.py"),
    os.path.join(REPO_ROOT, "visualization/circuit_top_acts.py"),
    os.path.join(REPO_ROOT, "visualization/circuit_analysis_builder_website.py")
]
for f in viz_files:
    patch_file(f, "strict=False)", "strict=False, weights_only=False)", "Visualization Fix")
edge_weights_file = os.path.join(REPO_ROOT, "visualization/get_edge_weights.py")
if not patch_file(edge_weights_file, "strict=False", "strict=False, weights_only=False", "Strategy A"):
    patch_file(edge_weights_file, "load_from_checkpoint(", "load_from_checkpoint(weights_only=False, ", "Strategy B")

# Load checkpoints correctly
clt_module_file = os.path.join(REPO_ROOT, "training/clt_module.py")
esm_colab_path = os.path.join(REPO_ROOT, "models/esm2_t6_8M_UR50D.pt")
if os.path.exists(clt_module_file):
    with open(clt_module_file, "r") as f:
        lines = f.readlines()
    new_lines = []
    patched_ghost = False
    target_line = "self._load_esm_weights(args.esm2_weight)"
    for line in lines:
        if target_line in line and not patched_ghost and "COLAB PATCH" not in line:
            indent = line.split("self")[0]
            # Injecting the fix block
            new_lines.append(f"{indent}# --- COLAB PATCH START ---\n")
            new_lines.append(f"{indent}import os\n")
            new_lines.append(f"{indent}if not os.path.exists(args.esm2_weight):\n")
            new_lines.append(f'{indent}    args.esm2_weight = "{esm_colab_path}"\n')
            new_lines.append(f"{indent}# --- COLAB PATCH END ---\n")
            new_lines.append(line)
            patched_ghost = True
        else:
            new_lines.append(line)
    if patched_ghost:
        with open(clt_module_file, "w") as f:
            f.writelines(new_lines)
else:
    print(f"Warning: Could not find {clt_module_file}")

### 1.5 Discover your own circuit (optional)
Use this section to train a probe and discover a circuit for your own custom dataset. We follow a similar protocol to Appendix D of the paper.

### **Input CSV format**
Your CSV must have two specific columns (not case-sensitive):
1.  **Sequence**: `sequence` or `mutated_sequence`.
2.  **Score**: `score`, `DMS_score`, or `class`

#### **Option A: Binary classification**
* **Sequence:** can vary in length.
* **Score:** Must contain **only** `0` and `1`.
* **Example:**
    ```csv
    sequence,class
    MKV...AAA,1
    AGL...TTV,0
    MKV...AAB,1
    ```

#### **Option B: Regression**
* **Sequence:** must be same length.
* **Score:** Continuous numbers (float).
* **Example:**
    ```csv
    mutant,mutated_sequence,DMS_score
    K3R,MSR...LYK,3.74
    K3Q,MSQ...LYK,3.75
    K3E,MSE...LYK,3.67
    ```

In [ ]:
# @title Run circuit discovery
# @markdown **Settings**
import os
import sys
import subprocess
import pty
import torch
import argparse
from google.colab import files
if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([argparse.Namespace])

# @markdown Select the type of task:
task_type = "Binary classification" # @param ["Binary classification", "Regression"]
# @markdown Check to upload your CSV file:
upload_csv = True # @param {type:"boolean"}
# @markdown Output folder name:
output_dir = "custom_circuit" # @param {type:"string"}
# @markdown Where to save the results:
external_path = "/content/experiments" # @param {type:"string"}

REPO_ROOT = "/content/ProtoMech"
SCRIPT_PATH = os.path.join(REPO_ROOT, "visualization", "auto_discover_circuit_website.py")

# --- 1. Input Handling ---
if upload_csv:
    print("Please upload your CSV file...")
    uploaded = files.upload()
    if not uploaded:
        sys.exit("Upload cancelled.")
    csv_name = list(uploaded.keys())[0]
    csv_path = os.path.join(os.getcwd(), csv_name)
else:
    sys.exit("Check 'upload_csv' to proceed.")
is_binary = (task_type == "Binary classification")
entry_name = os.path.splitext(csv_name)[0]
full_output_dir = os.path.join(external_path, output_dir)

# --- 2. Run Command ---
cmd = [
    "python", SCRIPT_PATH,
    "--csv_path", csv_path,
    "--is_binary", str(is_binary),
    "--output_dir", full_output_dir,
    "--entry_name", entry_name,
    "--clt_checkpoint", f"{REPO_ROOT}/models/CLT_L6_D3200/checkpoints/last.ckpt",
    "--esm_weights", f"{REPO_ROOT}/models/esm2_t6_8M_UR50D.pt",
    "--batch_size", "8"
]
print(f"\nStarting Discovery ({task_type})...")
print(f"   Input: {csv_name}")
print(f"   Output: {full_output_dir}/{entry_name}.json\n")
master, slave = pty.openpty()
p = subprocess.Popen(cmd, stdout=slave, stderr=slave, close_fds=True)
os.close(slave)
try:
    while True:
        try:
            data = os.read(master, 1024).decode()
            if not data: break
            sys.stdout.write(data)
            sys.stdout.flush()
        except OSError: break
except Exception: pass
p.wait()
os.close(master)

if p.returncode == 0:
    print(f"\nDone! Download your JSON here: {full_output_dir}/{entry_name}.json")
else:
    print("\nDiscovery Failed.")

In [ ]:
# @title 2. Add sequences to compute
# @markdown Example sequences:
# @markdown - Seq 1 (wildtype): `QYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE`
# @markdown - Seq 2: `QYKLILNGKTLKGETTTEAVDAWTAEKVFKQYANDNGVDGEWTYDDATKTFTVTE`
# @markdown - Seq 3: `QYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEQTYDDATKTFTVTE`

# @markdown Note: Use `Add Sequence +` to compare variations of the same protein (e.g., assessing the effect of mutations on a wildtype sequence). To discover a completely new circuit for a different protein, please reset and run the discovery pipeline again.
import ipywidgets as widgets
from IPython.display import display

class SequenceInputManager:
    def __init__(self):
        self.sequences = []
        self.container = widgets.VBox()
        self.add_button = widgets.Button(description="Add Sequence +", icon="plus")
        self.add_button.on_click(self.add_field)

        # Initial field
        self.add_field(None)

    def add_field(self, b):
        idx = len(self.sequences) + 1
        label = "Seq 1 (wildtype):" if idx == 1 else f"Seq {idx}:"
        text = widgets.Text(placeholder=f"Enter protein sequence {idx}...", layout=widgets.Layout(width='80%'))
        box = widgets.HBox([widgets.Label(value=label, layout=widgets.Layout(width='120px')), text])
        self.sequences.append(text)
        self.container.children = tuple(list(self.container.children) + [box])

    def get_sequences(self):
        return [w.value.strip() for w in self.sequences if w.value.strip()]

    def display(self):
        display(widgets.VBox([self.container, self.add_button]))

# Instantiate and display
seq_manager = SequenceInputManager()
seq_manager.display()

In [ ]:
# @title 3. Generate Circuit Files (Run after adding sequences)
# @markdown **Configuration**
# @markdown - `circuit` (optional): Leave empty to auto-generate from `Seq 1`. You can also find a list of pre-discovered circuits [here](https://github.com/amirgroup-codes/ProtoMech/blob/main/visualization/circuits.md).
# @markdown - `upload_custom_circuit`: Check to upload a custom JSON.
# @markdown - `output_dir`: Name of the folder to create.

import os
import sys
import subprocess
import shutil
from google.colab import files

# --- 1. Get Inputs from Widget ---
sequences = seq_manager.get_sequences()
circuit = "SPG1_STRSG_Olson_2014" # @param {type:"string"}
upload_custom_circuit = False # @param {type:"boolean"}
output_dir = "GB1" # @param {type:"string"}
external_path = "/content/experiments" # @param {type:"string"}

# --- Validation ---
if not sequences:
    print("❌ Error: No sequences provided. Please add sequences in the UI above.")
    sys.exit(1)
print(f"Processing {len(sequences)} sequences...")
print(f"   Seq 1 (wldtype): {sequences[0][:10]}...")

# --- Setup Paths ---
REPO_ROOT = "/content/ProtoMech"
SCRIPTS_DIR = os.path.join(REPO_ROOT, "visualization")
FULL_OUTPUT_DIR = os.path.join(external_path, output_dir)
os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)

if not os.path.exists(SCRIPTS_DIR):
    raise FileNotFoundError(f"Could not find directory: {SCRIPTS_DIR}")
os.chdir(SCRIPTS_DIR)
def run_realtime(command):
    """
    Runs a command with a pseudo-terminal to allow real-time
    output and progress bars in Colab.
    """
    master, slave = pty.openpty()
    p = subprocess.Popen(command, stdout=slave, stderr=slave, close_fds=True)
    os.close(slave)
    try:
        while True:
            try:
                data = os.read(master, 1024)
                if not data: break
            except OSError:
                break
    except Exception as e:
        pass #
    p.wait()
    os.close(master)
    return p.returncode, ""



circuit_json_path = None
needs_generation = False

# CASE A: User wants to upload a file
if upload_custom_circuit:
    print("\nPlease upload custom circuit json...")
    uploaded = files.upload()
    if not uploaded:
        print("Upload cancelled. Aborting.")
        sys.exit(1)
    filename = list(uploaded.keys())[0]
    target_path = os.path.join(FULL_OUTPUT_DIR, filename)
    os.rename(filename, target_path)
    circuit_json_path = target_path

# CASE B: User specified a circuit query
elif circuit.strip():
    clean_query = circuit.strip()
    json_filename = clean_query if clean_query.endswith(".json") else f"{clean_query}.json"
    base_name = os.path.splitext(clean_query)[0]

    if clean_query.startswith("IPR"):
        candidate_path = os.path.join(REPO_ROOT, "family_circuit/families/CLT_sequential", json_filename)
        if os.path.exists(candidate_path):
            circuit_json_path = candidate_path
        else:
            print(f"Warning: Could not find family circuit at {candidate_path}. Auto-generating...")
            needs_generation = True
    else:
        candidate_path = os.path.join(REPO_ROOT, "function_circuit/functions/CLT_sequential/multiples", base_name, "rand_multiples_fold0.json")
        if os.path.exists(candidate_path):
            circuit_json_path = candidate_path
        else:
            print(f"Warning: Could not find function file at {candidate_path}. Auto-generating...")
            needs_generation = True

# CASE C: No input provided
else:
    print("Auto-generating circuit from seq 1...")
    needs_generation = True



# --- Pipeline Execution ---
# Step 0: Generate Circuit (from Seq 1)
if needs_generation:
    print("\n[Step 0] Generating circuit JSON...")
    generated_json_path = os.path.join(FULL_OUTPUT_DIR, f"{output_dir}_circuit.json")
    cmd = [
        "python", "circuit_top_acts.py",
        "--sequence", sequences[0],
        "--output", generated_json_path
    ]
    exit_code, cmd_output = run_realtime(cmd)
    if exit_code != 0:
        print("Error generating circuit")
        circuit_json_path = None
    else:
        circuit_json_path = generated_json_path

# Step 1: Analyze All Sequences
if circuit_json_path:
    print("\n[Step 1] Running multi-sequence analysis...")
    if os.path.exists(output_dir):
        if os.path.islink(output_dir):
            os.unlink(output_dir)
        elif os.path.isdir(output_dir):
            shutil.rmtree(output_dir)
    os.symlink(FULL_OUTPUT_DIR, output_dir)
    cmd_analysis = [
        "python", "circuit_analysis_builder_website.py",
        "--entry_name", output_dir,
        "--circuit_json", circuit_json_path,
        "--sequences"
    ] + sequences
    exit_code, cmd_output = run_realtime(cmd_analysis)

    if exit_code != 0:
        print("Error conducting circuit analysis")
    else:
        print("\n[Step 2] Computing edge weights for each sequence (may take some time)...")

        subfolders = [f.path for f in os.scandir(FULL_OUTPUT_DIR) if f.is_dir() and "seq" in f.name]
        subfolders.sort() # Ensure seq1, seq2 order
        for folder_path in subfolders:
            folder_name = os.path.basename(folder_path)
            print(f"\n   Processing {folder_name}...")
            target_rel_path = os.path.join(output_dir, folder_name)
            cmd_weights = [
                "python", "get_edge_weights.py",
                "--base_folder", target_rel_path
            ]
            w_exit, w_out = run_realtime(cmd_weights)
            if w_exit != 0:
                print(f"   ❌ Failed to compute weights for {folder_name}")

        print("\n==========")
        print(f"Results saved to: {FULL_OUTPUT_DIR}")
        print("==========")
        print("Files generated:")
        for f in os.listdir(FULL_OUTPUT_DIR):
             print(f" - {f}")

    # Cleanup Symlink
    if os.path.islink(output_dir):
        os.unlink(output_dir)
else:
    if not needs_generation:
        print("Aborted: No valid circuit JSON found.")